# পাঠ ১৮: ক্রিপ্টোগ্রাফিক রসিদ দিয়ে AI এজেন্টদের নিরাপত্তা প্রদান

## হাতে-কলমে নোটবুক

এই নোটবুক চারটি অনুশীলনের মধ্য দিয়ে যায়:

১. **আপনার প্রথম রসিদে সাইন করুন** একটি এজেন্ট টুল কলের জন্য এবং যাচাই করুন।
২. **রসিদে জালবদল করুন** এবং যাচাই ব্যর্থ হতে দেখুন।
৩. **একটি তিনটি রসিদের চেইন গঠন করুন** এবং চেইনের অখণ্ডতা নিশ্চিত করুন।
৪. **মাইক্রোসফট এজেন্ট ফ্রেমওয়ার্ক টুল কল মোড়ান** যাতে প্রতিটি কাজ একটি রসিদ তৈরি করে।

সমস্ত ক্রিপ্টোগ্রাফিক প্রিমিটিভ ভালোভাবে রক্ষণাবেক্ষিত লাইব্রেরি থেকে আমদানি করা হয়েছে (`pynacl` Ed25519 এর জন্য, `jcs` RFC 8785 ক্যানোনিক্যাল JSON এর জন্য, `hashlib` পাইথনের স্ট্যান্ডার্ড লাইব্রেরি থেকে SHA-256 এর জন্য)। রসিদ লজিকটি নিজেই সরল পাইথন যা আপনি পড়তে এবং পরিবর্তন করতে পারবেন।

সেলগুলো যথাক্রমে চালান। প্রতিটি অধ্যায় স্বল্প এবং স্বনির্ভর।


## Setup

দুটি নির্ভরতাসমূহ ইনস্টল করুন। উভয়েরই অনুমতিপূর্ণ লাইসেন্স রয়েছে (Apache-2.0 / MIT)।


In [ ]:
!pip install -q pynacl jcs

In [ ]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import BadSignatureError
from jcs import canonicalize

## সহায়ক ইউটিলিটিজ

এই দুটি হেল্পার বেস64url এনকোডিং (প্যাডিং ছাড়া) এবং অ্যার্বিট্রারি অবজেক্টের SHA-256 হ্যাশিং পরিচালনা করে। এগুলি বাকী নোটবুকটিকে রসিদ লজিকেই ফোকাস রাখতে সাহায্য করে।


In [ ]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """Decode a base64url string that may be missing padding."""
    padding = "=" * ((4 - len(s) % 4) % 4)
    return base64.urlsafe_b64decode(s + padding)

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## Section 1: আপনার প্রথম রশিদে সাইন করুন

ভাবুন আমাদের **Contoso Travel** এজেন্ট সদ্য সিডনি থেকে লস অ্যাঞ্জেলেসে একটি গ্রাহকের জন্য উড়ানের তথ্য খুঁজে নিয়েছে। আমরা চাই এই টুল কলটি একটি সাইন করা রশিদ হিসেবে রেকর্ড করতে যাতে ভবিষ্যতের একটি নিরীক্ষক এটি আমাদের উপর বিশ্বাস না করেও যাচাই করতে পারে।

### Step 1.1: একটি সাইনিং কী তৈরি করুন

প্রোডাকশনে, এজেন্টের সাইনিং কী একটি হার্ডওয়্যার সিকিউরিটি মডিউল (HSM), Azure Key Vault, অথবা অনুরূপ সুরক্ষিত স্টোরে সংরক্ষিত থাকে। এই পাঠে আমরা একটি নতুন কী মেমোরিতে তৈরি করব।


In [ ]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

### Step 1.2: রসিদ পে-লোড তৈরি করুন

পে-লোডে সবকিছু থাকে যার মাধ্যমে রসিদটি প্রমাণ করতে চায়: কে কাজ করেছেন, কোন সরঞ্জামটি ব্যবহার করেছেন, কোন আর্গুমেন্ট দিয়ে, কী ফলাফল এসেছে, কোন নীতি অনুসারে, এবং কখন। আমরা আর্গুমেন্ট এবং ফলাফলগুলোকে সরাসরি অন্তর্ভুক্ত না করে হ্যাশ করি যাতে রসিদ সংবেদনশীল বিষয়বস্তু ফাঁস না করে।


In [ ]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

### Step 1.3: সাইন এবং রসিদ একত্রিত করুন

তিনটি ধাপ:

1. একই লজিক্যাল রসিদ উৎপাদন করে এমন দুটি ইমপ্লিমেন্টেশন বাইট-পরিচিতি বাইট তৈরি করার জন্য JCS ব্যবহার করে পে-লোডটিকে ক্যানোনিক্যালাইজ করুন।
2. ক্যানোনিক্যাল বাইটগুলি SHA-256 দিয়ে হ্যাশ করুন।
3. Ed25519 প্রাইভেট কী দিয়ে হ্যাশে স্বাক্ষর করুন।

তাহলে স্বাক্ষরটি মূল পে-লোডের সাথে সংযুক্ত করা হয় যাতে চূড়ান্ত রসিদ তৈরি হয়।


In [ ]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()
    signature_bytes = signing_key.sign(message_hash).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

### Step 1.4: রসিদ যাচাই করুন

যাচাই প্রক্রিয়াটি উল্টো। আমরা স্বাক্ষর সরিয়ে ফেলি, প্রামাণিক হ্যাশ পুনরায় গণনা করি, এবং রসিদে থাকা পাবলিক কী-এর সঙ্গে স্বাক্ষর পরীক্ষা করি।

এই যাচাই করতে একজন নিরীক্ষকের আমাদের থেকে কিছুই দরকার হয় না, শুধুমাত্র রসিদই যথেষ্ট। কোন সার্ভিস কল করার দরকার নেই, কোন কী ডিরেক্টরি অনুসন্ধান করার দরকার নেই, কোনো বিশ্বাসের প্রয়োজন নেই।


In [ ]:
def verify_receipt(receipt: dict) -> bool:
    """
    Verify a receipt's Ed25519 signature.
    Returns True if valid, False otherwise.
    """
    sig_obj = receipt.get("signature")
    if not sig_obj or sig_obj.get("alg") != "EdDSA":
        return False

    # Reconstruct the payload that was actually signed (everything except signature).
    payload = {k: v for k, v in receipt.items() if k != "signature"}

    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()

    try:
        verify_key = signing.VerifyKey(b64url_decode(sig_obj["public_key"]))
        verify_key.verify(message_hash, b64url_decode(sig_obj["sig"]))
        return True
    except BadSignatureError:
        return False
    except Exception as exc:
        print(f"Verification error: {exc}")
        return False

is_valid = verify_receipt(receipt)
print(f"Receipt is valid: {is_valid}")

আপনি দেখতে পাবেন `Receipt is valid: True`। এজেন্টটি তার প্রথম ক্রিপ্টোগ্রাফিক্যালি স্বাক্ষরিত অডিট রেকর্ড তৈরি করেছে।


## Section 2: রসিদে ছেদ ঘটানো

রসিদের পুরো উদ্দেশ্যই হলো এগুলো ছেদ-প্রমাণ। চলুন প্রমাণ করি।

আমরা রসিদের ঠিক এক অক্ষর পরিবর্তন করব এবং যাচাই ব্যর্থ হবে তা দেখব।


In [ ]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered)}")

### কি হলো ঠিকমতো?

যখন আমরা `policy_id` পরিবর্তন করলাম, তখন canonical বাইটগুলো পরিবর্তিত হলো। সেই বাইটগুলোর SHA-256 হ্যাশও পরিবর্তিত হলো। সিগনেচার (যা মূল হ্যাশের উপর ছিল) নতুন হ্যাশের সাথে আর মেলে না। যাচাই সঠিকভাবে `False` রিটার্ন করে।

রসিদে কোন ক্ষেত্র পরিবর্তন করার কোন উপায় নেই যেটি যাচাই করতে পারবে, যদি না আক্রমণকারী প্রাইভেট কীটি ধারণ করে। যতক্ষণ প্রাইভেট কীটি কি ভল্টে আছে এবং পাবলিক কী প্রকাশিত আছে, ছেঁড়াছেঁড়ি লুকানোর কোন উপায় নেই।

নিজেই চেষ্টা করুন: উপরের সেলে `tool_name` অথবা `agent_id` অথবা `timestamp` পরিবর্তন করুন এবং আবার চালান। প্রতিটি পরিবর্তন একটি অবৈধ রসিদ তৈরি করে।


## Section 3: একাধিক রসিদকে একত্রিত করা

একটি রসিদ একটি ক্রিয়াকলাপকে সুরক্ষিত করে। বেশিরভাগ এজেন্ট অনেকগুলি ক্রিয়াকলাপ সম্পাদন করে। পুরো ক্রমটিকে পরিবর্তন-প্রমাণ করে তুলতে, আমরা প্রতিটি রসিদকে পূর্ববর্তী রসিদের সাথে সংযুক্ত করি পূর্ববর্তী রসিদের হ্যাশ নতুন রসিদের পে-লোডে অন্তর্ভুক্ত করে।

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

যদি কেউ কোনো রসিদ সরিয়ে দেয় বা ক্রমবিন্যাস পরিবর্তন করে, তাহলে ঠিক ঐ স্থানে চেইন ভেঙে যায়। পরবর্তী যেকোনো রসিদের যাচাই ব্যর্থ হয় কারণ তার `previous_receipt_hash` আর তার পূর্ববর্তী রসিদের প্রকৃত হ্যাশের সাথে মেলে না।


In [ ]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [ ]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

In [ ]:
def verify_chain(chain: list) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt (except the genesis) must reference the previous receipt's hash.
      3. Sequence numbers must match each receipt's zero-based position in the chain.
    Returns a list of per-receipt result dicts.
    """
    results = []
    for i, receipt in enumerate(chain):
        sig_ok = verify_receipt(receipt)

        if i == 0:
            chain_ok = receipt["previous_receipt_hash"] is None
        else:
            expected = receipt_hash(chain[i - 1])
            chain_ok = receipt["previous_receipt_hash"] == expected

        seq_ok = receipt["sequence"] == i

        results.append({
            "index": i,
            "tool": receipt["tool_name"],
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok,
        })
    return results

for r in verify_chain(chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

এখন মাঝের রসিদের সাথে ছলনার মাধ্যমে চেইন ভাঙুন এবং পুনরায় যাচাই করুন। ছলিত রসিদ তার স্বাক্ষর পরীক্ষা ব্যর্থ করে, এবং পরবর্তী রসিদ তার চেইন-লিঙ্ক পরীক্ষা ব্যর্থ করে (কারণ তার `previous_receipt_hash` আর পরিবর্তিত মাঝের রসিদের হ্যাশের সাথে মিলে না)।


In [ ]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

রসিদ 0 এখনও যাচাইযোগ্য (এটি পরিবর্তিত হয়নি এবং নির্ভরশীল কোনো পূর্বসঙ্গী নেই)। রসিদ 1 এর সিগনেচার পরীক্ষা ব্যর্থ হয় কারণ আমরা `tool_args_hash` পরিবর্তন করেছি। রসিদ 2 এর চেইন-লিঙ্ক পরীক্ষা ব্যর্থ হয় কারণ এর `previous_receipt_hash` মূল (এখন পরিবর্তিত) রসিদ 1 এর বিরুদ্ধে গণনা করা হয়েছে।

যদিও একজন আক্রমণকারী পরিবর্তিত রসিদ 1 পুনঃস্বাক্ষর করতে পারেন (যা তারা ব্যক্তিগত চাবি ছাড়া করতে পারে না), রসিদ 2 এর চেইন-লিঙ্কের অসামঞ্জস্য তবুও বদলের তথ্য প্রমাণ করবে। পরিবর্তনটি লুকানোর জন্য, আক্রমণকারীকে পরিবর্তনের স্থান থেকে প্রতিটি রসিদ পুনঃস্বাক্ষর করতে হবে, যা ব্যক্তিগত চাবির অধিকার প্রয়োজন।


## Section 4: রিসিপ্ট স্বাক্ষরের সাথে একটি এজেন্ট টুল কল মোড়ানো

একটি বাস্তব স্থাপনে, আপনি চান না প্রতিটি এজেন্ট লেখক মনে রাখুক `make_receipt` কল করতে। আপনি চান প্রতিটি টুল আড্ডায় স্বয়ংক্রিয়ভাবে রিসিপ্ট স্বাক্ষর করা হোক।

এখানে সবচেয়ে সহজ প্যাটার্ন: একটি wrapper ক্লাস যা যেকোনো callable টুল ফাংশন নেয় এবং এর একটি রিসিপ্ট-জমা দেওয়া সংস্করণ ফিরিয়ে দেয়। এটি যেকোনো এজেন্ট ফ্রেমওয়ার্কের সাথে মানিয়ে নেয়, Microsoft Agent Framework (`agent_framework.azure`) সহ।

যদি আপনার Azure AI Foundry প্রকল্প সেটআপ না থাকে, তাহলে নিচের লোকাল মক প্যাটার্নটি এখনও প্রদর্শন করে।


In [ ]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [ ]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to AzureAIProjectAgentProvider as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")

### Microsoft Agent Framework এর সাথে ইন্টিগ্রেশন

উপরে উল্লেখিত `ReceiptedTool` র‍্যাপারটি ফ্রেমওয়ার্ক-অনিরপেক্ষ। এটিকে Microsoft Agent Framework দিয়ে নির্মিত একটি এজেন্টে ব্যবহারের জন্য, র‍্যাপ করা ফাংশনটিকে টুল হিসেবে নিবন্ধন করুন। একটি স্কেচ (আপনি মককে একটি বাস্তব Azure AI Foundry টুল নিবন্ধনের সাথে প্রতিস্থাপন করবেন):

```python
# ইন্টিগ্রেশন আকার দেখানোর ছদ্মকোড।
# from agent_framework.azure import AzureAIProjectAgentProvider
# from azure.identity import AzureCliCredential
#
# provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())
# agent = provider.create_agent(
#     নির্দেশাবলী="আপনি একটি কনটোসো ট্রাভেল এজেন্ট ...",
#     tools=[receipted_lookup],   # মোড়ানো টুল, কাঁচা ফাংশন নয়
# )
# response = agent.run("সিডনি থেকে লস এঞ্জেলেসে জুন মাসে ফ্লাইট খুঁজুন।")
#
# # রান করার পর, এজেন্ট যতো টুল কল করেছে তার প্রত্যেকটির একটি স্বাক্ষরযুক্ত রশিদ থাকবে:
# audit_chain = receipted_lookup.receipts
```

এজেন্ট ফ্রেমওয়ার্ককে রসিদ সম্পর্কে কিছু জানার প্রয়োজন নেই। রসিদ স্বাক্ষর টুলের চারপাশে র‍্যাপ করা হয়েছে, ফ্রেমওয়ার্কে না। এইভাবেই আপনি এজেন্টকে পুনর্লিখন না করে বিদ্যমান এজেন্ট কোডে উৎস যোগ করতে পারেন।


## সারসংক্ষেপ এবং স্ট্রেচ চ্যালেঞ্জ

আপনার কাছে রয়েছে:

- Ed25519 কী জোড়া তৈরি করা হয়েছে।
- একটি এজেন্ট টুল কলের জন্য রসিদ তৈরি এবং সই করা হয়েছে।
- শুধুমাত্র পাবলিক কী ব্যবহার করে রসিদ অফলাইনে যাচাই করা হয়েছে।
- একটি রসিদে বদল করা হয়েছে এবং যাচাই ব্যর্থ হয়েছে দেখা গেছে।
- তিনটি রসিদের একটি হ্যাশ-চেইন তৈরি করা হয়েছে।
- চেইনের মাঝখানে তদারকি করে সম্মিলিত স্বাক্ষর ব্যর্থতা এবং চেইন-লিঙ্ক ব্যর্থতা দেখা গেছে।
- একটি টুল ফাংশন স্বয়ংক্রিয় রসিদ স্বাক্ষর দিয়ে আবৃত করা হয়েছে।

**স্ট্রেচ চ্যালেঞ্জ।** রসিদ স্কিমাটিকে একটি `request_id` ফিল্ড (বিতরণকৃত ট্রেসিংয়ের জন্য UUID) দিয়ে সম্প্রসারিত করুন। `make_receipt`-এ এটি অন্তর্ভুক্ত করতে আপডেট করুন এবং নিশ্চিত করুন যে রসিদগুলি এখনও শেষ থেকে শেষ পর্যন্ত যাচাই হয়। তারপর স্বাক্ষর করার পর ফিল্ডটি পরিবর্তন করুন এবং যাচাই ব্যর্থ হয় তা নিশ্চিত করুন। এটি আপনাকে অভ্যন্তরীণভাবে বুঝতে বাধ্য করবে যে কীভাবে প্রতিটি বাইট canonical এনকোডিং-এর স্বাক্ষরের জন্য অবদান রাখে।

**গুরুত্বপূর্ণ সীমানা।** রসিদ তিনটি জিনিস প্রমাণ করে এবং মাত্র তিনটি: আতtributions (এই কী এই বিষয়বস্তুতে সই করেছে), অখণ্ডতা (সই করার পর বিষয়বস্তু পরিবর্তিত হয়নি), এবং ক্রমনির্দেশ (এই রসিদটি ঐ রসিদের পরে এসেছে)। এগুলি এজেন্টের কর্ম সঠিক ছিল, `policy_id`-তে নোঙরিত নীতি প্রকৃতপক্ষে মূল্যায়ন করা হয়েছে, বা এজেন্ট প্রত্যেক নিয়ম অনুসরণ করেছে তা প্রমাণ করে না। রসিদ হলো একটি ভিত্তি। শাসনব্যবস্থা হলো আপনি যা তার উপরে তৈরি করেন।

সেই সীমানা মাথায় রেখে লেসনের README আবার পড়ুন। রসিদের সাথে সর্বাধিক সাধারণ ভুল ধারণা হলো "আমাদের রসিদ আছে" অর্থ "আমরা শাসিত।" এটা নয়। রসিদ এজেন্টের আচরণ অডিটযোগ্য করে তোলে। তা সঠিক করে না।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**অস্বীকৃতি**:
এই নথিটি AI অনুবাদ পরিষেবা [Co-op Translator](https://github.com/Azure/co-op-translator) ব্যবহার করে অনূদিত হয়েছে। যদিও আমরা শুদ্ধতার জন্য চেষ্টা করি, অনুগ্রহ করে মনে রাখবেন যে স্বয়ংক্রিয় অনুবাদে ত্রুটি বা অসঙ্গতি থাকতে পারে। মূল নথিটি তার স্বভাষায় কর্তৃত্বপূর্ণ উৎস হিসেবে বিবেচিত হওয়া উচিত। গুরুত্বপূর্ণ তথ্যের জন্য পেশাদার মানব অনুবাদ সুপারিশ করা হয়। এই অনুবাদের ব্যবহারে প্রয়োজনীয় ভুল বোঝাবুঝি বা ভুল ব্যাখ্যার জন্য আমরা দায়বদ্ধ নই।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
